In [5]:
# This notebook builds a DrugCentral-shaped indication table from repurposed drug–disease pairs.
# It loads CSVs, compares to DrugCentral approved indications, resolves CAS and UMLS CUIs, then exports.
import pandas as pd  # DataFrames for tabular joins, filtering, and CSV I/O


In [ ]:
from pathlib import Path
import os

def resolve_primekg_root() -> Path:
    if env := os.environ.get("PRIMEKG_ROOT"):
        root = Path(env).expanduser().resolve()
        if (root / "datasets" / "data" / "drugcentral").is_dir():
            return root
        raise FileNotFoundError(f"PRIMEKG_ROOT invalid: {root}")
    for start in (Path.cwd().resolve(),):
        for parent in (start, *start.parents):
            if (parent / "datasets" / "data" / "drugcentral").is_dir():
                return parent
            if parent.name == "repurposed_drug" and (parent.parent.parent / "datasets" / "data" / "drugcentral").is_dir():
                return parent.parent.parent
            if parent.name == "PrimeKG-Plus_release" and (parent.parent / "datasets" / "data" / "drugcentral").is_dir():
                return parent.parent
    raise FileNotFoundError("Set PRIMEKG_ROOT to the PrimeKG repo root (contains datasets/data/).")

PROJECT_ROOT = resolve_primekg_root()
DATA_DIR = PROJECT_ROOT / "datasets" / "data"
PREP_DIR = Path.cwd()  # run notebook with CWD = source_prep/repurposed_drug/


In [2]:
# List files in the current working directory whose names contain the repurposed-drug CSV filename.
!ls | grep inputs/RepurposedDrug.csv  # Shell magic: confirms the input file is present before pd.read_csv


inputs/RepurposedDrug.csv


In [3]:
# Preview the first rows of the repurposed pairs file (columns should include Drug_name, Disease_name).
!head inputs/RepurposedDrug.csv  # Unix head: quick schema/content check without loading the full file into RAM


﻿Drug_name,Disease_name
Acetylcysteine,"Kidney Failure, Acute"
Acitretin,Psoriasis
Aclidinium Bromide,Chronic Obstructive Airway Disease
Adapalene,Acne Vulgaris
Adefovir Dipivoxil,Hepatitis b
Adefovir Dipivoxil,"Hepatitis B, Chronic"
Adefovir,Hepatitis b
Adefovir,"Hepatitis B, Chronic"
Adenosine,Atherosclerosis


In [4]:
# Load all repurposed candidate rows from disk into a DataFrame (one row ≈ one proposed drug–disease use).
df = pd.read_csv(PREP_DIR / "inputs/RepurposedDrug.csv")  # Default comma separator; paths are relative to the notebook CWD
df.head(3)  # Jupyter displays the first 3 rows so you can verify parsing and column names


,Drug_name,Disease_name
0,Acetylcysteine,"Kidney Failure, Acute"
1,Acitretin,Psoriasis
2,Aclidinium Bromide,Chronic Obstructive Airway Disease


In [7]:
len(df)

656

In [6]:
df[df.Drug_name=="Ibrexafungerp"]

,Drug_name,Disease_name


In [8]:
df[df.Drug_name=="ibrexafungerp"]

,Drug_name,Disease_name


In [3]:
# Collect distinct drug and disease strings from the repurposed file (order not guaranteed; used for later stats).
rep_drugs = list(df.Drug_name.unique())  # Unique drug labels as Python strings
rep_diseases = list(df.Disease_name.unique())  # Unique disease labels as Python strings


### Phase 4 — DrugCentral as the approved-indication source (not PrimeKG edges)

- **Repurposed vs approved:** Compare repurposed candidate pairs to DrugCentral **indications** using `df_drug_central`: drug `name`, disease `concept_name`, relation `relationship_name` (analogous to legacy KG `x_name` / `y_name` / `relation`).
- **CAS for new pairs:** Resolve drug CAS primarily from **DrugBank XML** (name/synonym); if missing, fall back to DrugCentral **`cas_reg_no`** via `_dc_cas_by_lower_name` (built from the `indication` table).
- **Disease → UMLS CUI (hybrid):** `resolve_umls_for_repurposed_disease` — **offline** over full UMLS ENG + MRSTY with tie-break **MTH preferred name (PN)** → source-level `is_preferred` → semantic-type rank; if still `None`, call **UMLS UTS** (`pip install requests`, env **`UMLS_API_KEY`**).

**Run order:** imports / pandas → repurposed + DrugCentral cells → DrugBank cell (imports `csv`, `re`, `os`, `requests`) → UMLS path cell → MRSTY + UMLS index cells → resolver cell → `split_pair` / `pair_ids` loop.



In [4]:
# -----------------------------------------------------------------------------
# DrugCentral cleaned export: treat as the reference set of *approved* drug–disease indications (not from PrimeKG).
# Expect a *_cleaned.csv produced by clean_drugcentral_drug_disease_csv.py (heavy mol columns already removed).
# -----------------------------------------------------------------------------
DC_CLEANED = DATA_DIR / "drugcentral/05Oct2023_drug_disease_cleaned.csv"  # Absolute path to the snapshot you are auditing
df_drug_central = pd.read_csv(DC_CLEANED, low_memory=False)  # low_memory=False avoids mixed-type column inference warnings on wide files

# Keep only rows whose relation label is the OMOP/DrugCentral "indication" relationship (approved use).
indication = df_drug_central.loc[df_drug_central["relationship_name"] == "indication"].copy()  # .copy() avoids SettingWithCopyWarning on later edits
len(indication)  # Display row count as a quick sanity check after filtering


11775

In [5]:
# Inspect column names and sample values: expect drug `name`, disease `concept_name`, identifiers like `cas_reg_no`, `umls_cui`, …
indication.head(3)  # First 3 rows for visual validation of the filtered indication slice


,cd_id,cd_formula,cd_molweight,id,clogp,alogs,cas_reg_no,tpsa,lipinski,name,...,status,id.1,struct_id,concept_id,relationship_name,concept_name,umls_cui,snomed_full_name,cui_semantic_type,snomed_conceptid
2,1,NO,30.006,1948,0.74,NaN,10102-43-9,34.14,NaN,nitric oxide,...,NaN,152152,1948,21013731,indication,Persistent pulmonary hypertension of the newborn,C0031190,Persistent pulmonary hypertension of the newborn,T019,233815004.0
4,1,NO,30.006,1948,0.74,NaN,10102-43-9,34.14,NaN,nitric oxide,...,NaN,172749,1948,21001478,indication,Pulmonary hypertension,C0020542,Pulmonary hypertension,T047,70995007.0
5,2,CH2O,30.026,3244,0.35,0.82,50-00-0,17.07,0.0,formaldehyde,...,NaN,172446,3244,21001828,indication,Influenza,C0021400,Influenza,T047,6142004.0


In [6]:
# Collapse duplicate indication rows that share the same drug name and disease concept name (keep first occurrence).
indication = indication.drop_duplicates(subset=["name", "concept_name"])  # subset= defines the business key for "one approved pair"
len(indication)  # New row count after deduplication


11775

In [7]:
# Build canonical string keys: lowercase_drug + "_" + lowercase_disease (must stay consistent with split_pair below).
rep_pairs = (df["Drug_name"].str.lower() + "_" + df["Disease_name"].str.lower()).tolist()  # One key per repurposed table row
ind_pairs = (indication["name"].str.lower() + "_" + indication["concept_name"].str.lower()).tolist()  # Keys for approved indications


In [8]:
# Find repurposed pair keys that already appear as an approved indication in DrugCentral (overlap / not novel).
common_pairs = [p for p in rep_pairs if p in ind_pairs]  # List comprehension filters against the approved set


In [9]:
# How many repurposed rows are already "approved" in DrugCentral — useful overlap metric.
len(common_pairs)  # Integer count displayed in notebook output


125

In [10]:
# Novel repurposed pairs: present in the repurposed file but absent from the DrugCentral indication key set.
new_pairs = [p for p in rep_pairs if not p in ind_pairs]  # Negated membership → candidates for CAS/CUI resolution


In [11]:
# Number of novel pairs that still need DrugBank/DrugCentral CAS and UMLS CUI mapping.
len(new_pairs)  # Drives expected runtime of the resolution loop


531

In [12]:
# Show the first few novel pair keys for spot-checking string formatting (underscore split, lowercasing).
new_pairs[:5]  # Python slice: first five elements


['acetylcysteine_kidney failure, acute',
 'acitretin_psoriasis',
 'aclidinium bromide_chronic obstructive airway disease',
 'adefovir dipivoxil_hepatitis b',
 'adefovir dipivoxil_hepatitis b, chronic']

In [13]:
# =============================================================================
# DrugBank XML: build a lookup from lowercased primary name and synonyms to CAS registry number and DrugBank id.
# Also imports stdlib modules used here and in later UMLS cells; `requests` is optional (UTS API fallback).
# One-time parse of a large XML can take on the order of minutes depending on disk/CPU.
# =============================================================================
import csv  # Later: DictReader on the UMLS English extract
import os  # Read UMLS_API_KEY from environment for UTS calls
import re  # Normalize disease strings and parse TUI patterns from UTS JSON
import xml.etree.ElementTree as ET  # Incremental parse of DrugBank XML

try:
    import requests  # HTTP client for UMLS UTS REST when offline resolution fails
except ImportError:
    requests = None  # Defer network fallback until user installs requests via pip

DRUGBANK_XML = DATA_DIR / "drugbank/Nurset_data_drugbank/2025_01_ver_full_database.xml"  # Full DrugBank dump path
DB_NS = "http://www.drugbank.ca"  # Default namespace URI for DrugBank XML elements


def _q(tag: str) -> str:
    """Expand a local tag name into a Clark notation {namespace}tag for ElementTree comparisons."""
    return f"{{{DB_NS}}}{tag}"  # ET uses expanded names when the XML defines a default xmlns


def build_drugbank_name_lookup(xml_path: str) -> dict:
    """Stream-parse DrugBank XML; return dict[lower_name] -> {cas_reg_no, drugbank_id} (first-seen wins unless filling blanks)."""
    lookup = {}  # Outer map from normalized alias string to identifier dict

    def remember(alias: str, cas: str | None, db_id: str | None) -> None:
        """Register one alias; do not overwrite existing CAS/DB id unless the slot is empty."""
        k = (alias or "").strip().lower()  # Normalization key for case-insensitive name matching
        if not k:
            return  # Skip empty synonyms
        if k not in lookup:
            lookup[k] = {"cas_reg_no": cas, "drugbank_id": db_id}  # Initialize record for first alias
        elif cas and not lookup[k].get("cas_reg_no"):
            lookup[k]["cas_reg_no"] = cas  # Backfill CAS from a later synonym if primary had none
        if db_id and not lookup[k].get("drugbank_id"):
            lookup[k]["drugbank_id"] = db_id  # Backfill primary DrugBank id similarly

    for _, elem in ET.iterparse(xml_path, events=("end",)):  # iterparse avoids loading entire tree into memory
        if elem.tag != _q("drug"):
            continue  # Only process closing </drug> events
        db_id = None  # Will hold primary DrugBank accession
        name = None  # Canonical drug name element
        cas = None  # CAS registry number string if present
        synonyms_el = None  # Reference to <synonyms> subtree if present
        for child in elem:  # Single pass over immediate children of <drug>
            if child.tag == _q("drugbank-id") and child.attrib.get("primary") == "true":
                db_id = (child.text or "").strip() or None  # Primary id only
            elif child.tag == _q("name"):
                name = (child.text or "").strip() or None  # Official name text
            elif child.tag == _q("cas-number"):
                t = (child.text or "").strip()
                cas = t if t else None  # Normalize empty CAS to None
            elif child.tag == _q("synonyms"):
                synonyms_el = child  # Defer synonym iteration until name is known
        if name:
            remember(name, cas, db_id)  # Always index the primary name
            if synonyms_el is not None:
                for syn in synonyms_el.findall(_q("synonym")):
                    remember((syn.text or "").strip(), cas, db_id)  # Each synonym shares same CAS/db_id
        elem.clear()  # Drop subtree to cap memory during iterparse
    return lookup  # Completed alias → identifiers map


drugbank_by_name = build_drugbank_name_lookup(DRUGBANK_XML)  # Execute the one-time build for this notebook session


In [14]:
type(drugbank_by_name)  # Expect dict; confirms the lookup object class after parsing


dict

In [15]:
drugbank_by_name  # Jupyter pretty-prints a large dict; can be slow — use sparingly in production


{'lepirudin': {'cas_reg_no': '138068-37-8', 'drugbank_id': 'DB00001'},
 'phylloquinone': {'cas_reg_no': '84-80-0', 'drugbank_id': 'DB01022'},
 'calcium': {'cas_reg_no': '7440-70-2', 'drugbank_id': 'DB01373'},
 '[leu1, thr2]-63-desulfohirudin': {'cas_reg_no': '138068-37-8',
  'drugbank_id': 'DB00001'},
 'desulfatohirudin': {'cas_reg_no': '138068-37-8', 'drugbank_id': 'DB00001'},
 'hirudin variant-1': {'cas_reg_no': '138068-37-8', 'drugbank_id': 'DB00001'},
 'lepirudin recombinant': {'cas_reg_no': '138068-37-8',
  'drugbank_id': 'DB00001'},
 'r-hirudin': {'cas_reg_no': '138068-37-8', 'drugbank_id': 'DB00001'},
 'cetuximab': {'cas_reg_no': '205923-56-4', 'drugbank_id': 'DB00002'},
 'cétuximab': {'cas_reg_no': '205923-56-4', 'drugbank_id': 'DB00002'},
 'cetuximabum': {'cas_reg_no': '205923-56-4', 'drugbank_id': 'DB00002'},
 'dornase alfa': {'cas_reg_no': '143831-71-4', 'drugbank_id': 'DB00003'},
 'deoxyribonuclease (human clone 18-1 protein moiety)': {'cas_reg_no': '143831-71-4',
  'drugba

In [16]:
# -----------------------------------------------------------------------------
# UMLS resource paths and configuration: MRSTY gives CUI→TUI; English CSV gives strings per CUI.
# This pipeline does not use MONDO cross-references here — only UMLS files + optional UTS API.
# -----------------------------------------------------------------------------
MRSTY_PATH = DATA_DIR / "umls/2025AB-full/META/2025AB/META/MRSTY.RRF"  # Semantic type assignments file
UMLS_ENG_CSV = DATA_DIR / "umls/umls_2025AB.csv"  # Project-specific English string extract

# TUIs considered "disease-like" for repurposed indication alignment (subset of UMLS semantic groups).
# T019 congenital; T020 acquired abnormality; T037 injury/poisoning; T046 pathologic function;
# T047 disease or syndrome; T048 mental or behavioral; T191 neoplastic process.
ALLOWED_DISEASE_STYS = frozenset({"T019", "T020", "T037", "T046", "T047", "T048", "T191"})  # frozenset for fast intersection tests

# When several CUIs tie on lexical features, prefer the CUI whose types appear earlier in this list (project heuristic).
# Smaller index in _sty_tiebreak_rank_for means higher priority.
_STY_TIEBREAK_ORDER = ["T047", "T191", "T048", "T019", "T020", "T046", "T037"]  # Ordered list used as tie-break ladder


In [17]:
# MRSTY.RRF is pipe-delimited; first field is CUI, second is TUI (verify against your UMLS release).
!head -n3 "$MRSTY_PATH"  # set MRSTY_PATH in prior cell  # Show first 3 physical lines


C0000005|T116|||||
C0000005|T121|||||
C0000005|T130|||||


In [18]:
# -----------------------------------------------------------------------------
# Step A (offline): load every CUI→TUI pair from MRSTY into memory for fast disease-type filtering.
# Any CUI that appears in the English strings file can be checked against ALLOWED_DISEASE_STYS.
# -----------------------------------------------------------------------------
_cui_tuis_full: dict[str, set[str]] = {}  # CUI -> set of all TUIs asserted for that concept in MRSTY
with open(MRSTY_PATH, encoding="utf-8", errors="replace") as _mf:  # errors=replace avoids decode crashes on odd bytes
    for _line in _mf:  # Line-by-line streaming read
        _parts = _line.split("|")  # UMLS RRF family uses pipe separators
        if len(_parts) < 2:
            continue  # Malformed line guard
        _cui, _tui = _parts[0], _parts[1]  # Primary key fields for MRSTY
        _cui_tuis_full.setdefault(_cui, set()).add(_tui)  # Accumulate multiple TUIs per CUI


In [19]:
# Debug convenience: the full map is huge — in pipelines prefer len(_cui_tuis_full) or sampling keys.
_cui_tuis_full  # Jupyter displays structure; example shape commented in original notebook


{'C0000005': {'T116', 'T121', 'T130'},
 'C0000039': {'T109', 'T121'},
 'C0000052': {'T116', 'T126'},
 'C0000074': {'T109'},
 'C0000084': {'T116', 'T123'},
 'C0000096': {'T109', 'T121'},
 'C0000097': {'T109', 'T131'},
 'C0000098': {'T109', 'T131'},
 'C0000102': {'T109', 'T131'},
 'C0000103': {'T109', 'T130', 'T131'},
 'C0000107': {'T116', 'T121'},
 'C0000119': {'T109', 'T125'},
 'C0000120': {'T109', 'T121'},
 'C0000132': {'T116', 'T126'},
 'C0000137': {'T114', 'T123'},
 'C0000139': {'T109', 'T121'},
 'C0000151': {'T109', 'T121'},
 'C0000152': {'T116', 'T126'},
 'C0000163': {'T109', 'T125'},
 'C0000165': {'T116', 'T126'},
 'C0000167': {'T109', 'T125'},
 'C0000172': {'T109', 'T121'},
 'C0000173': {'T109', 'T121'},
 'C0000176': {'T109', 'T130'},
 'C0000184': {'T116', 'T126'},
 'C0000189': {'T116', 'T126'},
 'C0000190': {'T114', 'T121'},
 'C0000194': {'T109', 'T130'},
 'C0000204': {'T109'},
 'C0000215': {'T109', 'T131'},
 'C0000220': {'T109', 'T131'},
 'C0000232': {'T109', 'T130'},
 'C00002

In [ ]:
# Print the resolved path string so rerunning cells out of order still shows which file Step B will read.
UMLS_ENG_CSV  # Last expression echoes the variable value


In [21]:
umls_df = pd.read_csv(UMLS_ENG_CSV)  # Load English extract into pandas for exploratory profiling
umls_df.head(2)  # Peek at two rows to see columns like cui, language, source_name, is_preferred


/var/folders/rh/fl3hq65n6_dbw6bssnkh01r80000gn/T/ipykernel_43338/1635966915.py:1: DtypeWarning: Columns (9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  umls_df = pd.read_csv("/Users/ljw303/YANG_DATA/PrimeKG/datasets/data/umls/umls_2025AB.csv")


,cui,language,term_status,lui,string_type,string_identifier,is_preferred,aui,source_aui,source_cui,source_descriptor_dui,source,source_term_type,source_code,source_name,x1,x2,x3,x4
0,C0000005,ENG,P,L0000005,PF,S0007492,Y,A26634265,NaN,M0019694,D012711,MSH,PEP,D012711,(131)I-Macroaggregated Albumin,0,N,NaN,\n
1,C0000005,ENG,S,L0270109,PF,S0007491,Y,A26634266,NaN,M0019694,D012711,MSH,ET,D012711,(131)I-MAA,0,N,NaN,\n


In [23]:
umls_df.term_status.unique()  # Which term status flags exist in this extract (data QA)


array(['P', 'S'], dtype=object)

In [24]:
umls_df.string_type.unique()  # Which string_type values appear (e.g. PT, SY)


array(['PF', 'VO'], dtype=object)

In [25]:
umls_df.source.unique()  # Which vocab sources contributed rows (can be many)


array(['MSH', 'RXNORM', 'MTH', 'SNMI', 'LNC', 'SNOMEDCT_US', 'SNM', 'CSP',
       'PSY', 'CHV', 'RCD', 'LCH_NW', 'AOD', 'NCI', 'PDQ', 'CPM', 'GO',
       'LCH', 'MMSL', 'MEDCIN', 'MTHSPL', 'NDDF', 'ATC', 'USPMG', 'USP',
       'GS', 'VANDF', 'DRUGBANK', 'MEDLINEPLUS', 'CST', 'UWDA', 'FMA',
       'HL7V2.5', 'OMIM', 'ICPC2ICD10ENG', 'ICD10', 'ICD10AM', 'MDR',
       'RCDSY', 'ICD10CM', 'HPO', 'WHO', 'CCPSS', 'COSTAR', 'NOC',
       'MTHICD9', 'ICPC2P', 'NANDA-I', 'DXP', 'SNOMEDCT_VET', 'ICPC',
       'ICPC2EENG', 'RCDAE', 'ICNP', 'CCS', 'ICD9CM', 'BI', 'NEU',
       'ORPHANET', 'PCDS', 'NCBI', 'MED-RT', 'HL7V3.0', 'UMD', 'QMR',
       'RAM', 'SPN', 'AIR', 'CCC', 'ICD10AMAE', 'ICD10AE', 'MTHMST',
       'DSM-5', 'NIC', 'ALT', 'RCDSA', 'ICF', 'ICF-CY', 'CCSR_ICD10PCS',
       'OMS', 'CPT', 'JABL', 'NUCCHCPT', 'CDCREC', 'HCPT', 'HCPCS',
       'CCSR_ICD10CM', 'MTHICPC2EAE', 'CDT', 'PNDS', 'CVX', 'ICD10PCS',
       'MCM', 'MTHICPC2ICD10AE', 'HGNC', 'HCDT', 'MTHCMSFRF', 'AOT',
       'ULT', 

In [20]:
# CSV header line + a few data rows: confirms delimiter and column order for the manual index build below.
!head -n4 "$UMLS_ENG_CSV"  # Four lines: header + 3 records


cui,language,term_status,lui,string_type,string_identifier,is_preferred,aui,source_aui,source_cui,source_descriptor_dui,source,source_term_type,source_code,source_name,x1,x2,x3,x4
C0000005,ENG,P,L0000005,PF,S0007492,Y,A26634265,,M0019694,D012711,MSH,PEP,D012711,(131)I-Macroaggregated Albumin,0,N,,"
"
C0000005,ENG,S,L0270109,PF,S0007491,Y,A26634266,,M0019694,D012711,MSH,ET,D012711,(131)I-MAA,0,N,,"


In [26]:
# -----------------------------------------------------------------------------
# Step B (offline): scan ENG rows once to build three indexes used by the resolver:
#   _cui_mth_pn — Metathesaurus preferred name (source=MTH, term type PN) per CUI
#   _cui_pref_names_wide — all normalized strings marked is_preferred=Y per CUI
#   _name_to_cuis_wide — normalized lowercase string → list of CUIs (deduped order preserved)
# -----------------------------------------------------------------------------
_cui_pref_names_wide: dict[str, set[str]] = {}  # CUI -> set of preferred English strings (any source)
_cui_mth_pn: dict[str, str] = {}  # CUI -> single MTH preferred normalized string (first wins)
_name_to_cuis_wide: dict[str, list[str]] = {}  # Normalized name -> CUIs that carry that string
with open(UMLS_ENG_CSV, encoding="utf-8", errors="replace", newline="") as _uf:  # newline="" lets csv module handle newlines inside fields
    _ur = csv.DictReader(_uf)  # Row dicts keyed by header names
    for _row in _ur:  # Iterate entire English release subset file
        if _row.get("language") != "ENG":
            continue  # Non-English rows skipped to match resolver assumptions
        _cui = (_row.get("cui") or "").strip()  # UMLS concept unique identifier
        if not _cui:
            continue  # Skip rows without CUI
        _raw = _row.get("source_name") or ""  # Raw surface string from the extract
        _nm = re.sub(r"\s+", " ", _raw.strip().lower())  # Collapse whitespace; lower-case to match query normalization
        if not _nm:
            continue  # Skip empty strings after normalization
        if _row.get("source") == "MTH" and _row.get("source_term_type") == "PN":
            if _cui not in _cui_mth_pn:
                _cui_mth_pn[_cui] = _nm  # Record first-seen MTH PN per CUI
        if _row.get("is_preferred") == "Y":
            _cui_pref_names_wide.setdefault(_cui, set()).add(_nm)  # Track all preferred synonyms
        _name_to_cuis_wide.setdefault(_nm, []).append(_cui)  # Inverted index: name -> CUIs

for _k, _lst in list(_name_to_cuis_wide.items()):  # Materialize items() to allow dict mutation pattern
    _name_to_cuis_wide[_k] = list(dict.fromkeys(_lst))  # Dedupe CUIs while preserving first-seen order


In [27]:
_name_to_cuis_wide  # Large inverted index; display only for debugging


{'(131)i-macroaggregated albumin': ['C0000005'],
 '(131)i-maa': ['C0000005'],
 '1,2-dipalmitoylphosphatidylcholine': ['C0000039'],
 '1,2 dipalmitoylphosphatidylcholine': ['C0000039'],
 '1,2-dihexadecyl-sn-glycerophosphocholine': ['C0000039'],
 '1,2 dihexadecyl sn glycerophosphocholine': ['C0000039'],
 '1,2-dipalmitoyl-glycerophosphocholine': ['C0000039'],
 '1,2 dipalmitoyl glycerophosphocholine': ['C0000039'],
 'dipalmitoylphosphatidylcholine': ['C0000039', 'C0216971'],
 'dipalmitoylglycerophosphocholine': ['C0000039'],
 'dipalmitoyllecithin': ['C0000039'],
 '3,5,9-trioxa-4-phosphapentacosan-1-aminium, 4-hydroxy-n,n,n-trimethyl-10-oxo-7-((1-oxohexadecyl)oxy)-, inner salt, 4-oxide': ['C0000039'],
 'dipalmitoylphosphatidylcholine (substance)': ['C0000039'],
 'dipalmitoyl phosphatidylcholine': ['C0000039'],
 'phosphatidylcholine, dipalmitoyl': ['C0000039'],
 'dppc': ['C0000039'],
 '1,4-alpha-glucan branching enzyme': ['C0000052', 'C1415001'],
 '1,4 alpha glucan branching enzyme': ['C00000

In [28]:
_cui_mth_pn  # CUI -> MTH preferred name map; large — use len() or samples in scripts


{'C0000039': '1,2-dipalmitoylphosphatidylcholine',
 'C0000052': '1,4-alpha-glucan branching enzyme',
 'C0000084': '1-carboxyglutamic acid',
 'C0000163': '17-hydroxycorticosteroids',
 'C0000165': '17-hydroxysteroid dehydrogenases',
 'C0000189': '2-5a synthetase',
 'C0000215': '2,4,5-trichlorophenoxyacetic acid',
 'C0000257': '2-aminoadipic acid',
 'C0000268': '2-bromolysergic acid diethylamide',
 'C0000291': '2-isopropylmalate synthase',
 'C0000294': 'mesna',
 'C0000325': '20-methylcholanthrene',
 'C0000334': '24,25-dihydroxyvitamin d 3',
 'C0000340': 'calcidiol 1-monooxygenase',
 'C0000343': '25-hydroxyvitamin d 2',
 'C0000359': "3',5'-cyclic-nucleotide phosphodiesterase",
 'C0000370': '3,3-dichlorobenzidine',
 'C0000376': '3,4-dihydroxyphenylacetic acid',
 'C0000378': 'droxidopa',
 'C0000379': '3,4-methylenedioxyamphetamine',
 'C0000392': 'beta-alanine',
 'C0000404': '3-hydroxyacyl coa dehydrogenases',
 'C0000413': '3-hydroxysteroid dehydrogenases',
 'C0000464': 'docosahexaenoate',
 '

In [ ]:
# Preferred-name sets per CUI; membership tests drive resolver scoring.
_cui_pref_names_wide  # dict[CUI, set[str]]; can be very large


In [ ]:
# =============================================================================
# Disease string normalization, UTS REST helpers, and hybrid UMLS resolver (see following cells).
# =============================================================================
# --- Extended guide (read before editing) ------------------------------------
# What this section does
#   Map the disease side of each repurposed pair to one UMLS CUI so rows resemble DrugCentral identifiers.
#
# Prerequisites from earlier cells
#   _cui_tuis_full, _name_to_cuis_wide, _cui_mth_pn, _cui_pref_names_wide,
#   ALLOWED_DISEASE_STYS, _STY_TIEBREAK_ORDER.
#
# Two-stage resolution
#   1) Offline over local UMLS ENG + MRSTY only.
#   2) If no CUI, call UTS search/current using UMLS_API_KEY in the environment.
#
# Normalization contract
#   _norm_disease_label must match how keys were inserted into _name_to_cuis_wide (whitespace + case).
#
# Console tags during the main loop (later cell)
#   [ENDS_ES] / [ENDS_S] — pluralization QA hints; [UMLS_API] — CUI came from UTS not offline index.
#
# Security
#   export UMLS_API_KEY="..." locally; never commit API keys to git.
# -----------------------------------------------------------------------------


In [30]:
def _norm_disease_label(s: str) -> str:
    """Normalize free-text disease labels for lookup in _name_to_cuis_wide (whitespace collapsed, lowercased)."""
    return re.sub(r"\s+", " ", (s or "").strip().lower())  # None-safe: treat as empty string before strip

tests = [  # Small hand-picked examples to validate normalization behavior in-notebook
    "  Acute   Myocardial Infarction ",
    "Diabetes mellitus",
    "   Chronic   kidney disease   ",
    "",
    None
]

for t in tests:  # Iterate each test input
    print(f"Input: {t!r}")  # Show raw input including whitespace
    print(f"Output: {_norm_disease_label(t)!r}")  # Show normalized form used for dictionary keys
    print("---")  # Visual separator between cases


Input: '  Acute   Myocardial Infarction '
Output: 'acute myocardial infarction'
---
Input: 'Diabetes mellitus'
Output: 'diabetes mellitus'
---
Input: '   Chronic   kidney disease   '
Output: 'chronic kidney disease'
---
Input: ''
Output: ''
---
Input: None
Output: ''
---


In [29]:
_STY_TIEBREAK_ORDER  # Echo the tie-break ladder (same object as defined in the UMLS paths cell)


['T047', 'T191', 'T048', 'T019', 'T020', 'T046', 'T037']

In [32]:
def _sty_tiebreak_rank_for(tuis: set | frozenset) -> int:
    """Return the best (smallest) priority index among TUIs present; len(order) means no known TUI matched."""
    for i, p in enumerate(_STY_TIEBREAK_ORDER):  # Walk priority list in order
        print(i, " --> ", p)  # Debug print: shows enumeration progress (noisy but useful when developing tie-break)
        if p in tuis:
            return i  # First matching high-priority TUI wins
    return len(_STY_TIEBREAK_ORDER)  # Fallback rank worse than any explicit match

tests = [  # Synthetic TUI sets to exercise tie-break branches
    {"T047"},                         # Single disease/syndrome type
    {"T048", "T191"},                # Multiple types — expect preferred order from list
    {"T020", "T046"},                # Mid-list types
    {"T999"},                       # Unknown TUI — should fall through to default rank
    set(),                          # Empty TUI set
    {"T037", "T191"}                # Order between injury and neoplasm per _STY_TIEBREAK_ORDER
]

for t in tests:  # Run each synthetic case
    print(f"TUIs: {t}")
    print(f"Rank: {_sty_tiebreak_rank_for(t)}")
    print("---")


TUIs: {'T047'}
0  -->  T047
Rank: 0
---
TUIs: {'T191', 'T048'}
0  -->  T047
1  -->  T191
Rank: 1
---
TUIs: {'T046', 'T020'}
0  -->  T047
1  -->  T191
2  -->  T048
3  -->  T019
4  -->  T020
Rank: 4
---
TUIs: {'T999'}
0  -->  T047
1  -->  T191
2  -->  T048
3  -->  T019
4  -->  T020
5  -->  T046
6  -->  T037
Rank: 7
---
TUIs: set()
0  -->  T047
1  -->  T191
2  -->  T048
3  -->  T019
4  -->  T020
5  -->  T046
6  -->  T037
Rank: 7
---
TUIs: {'T037', 'T191'}
0  -->  T047
1  -->  T191
Rank: 1
---


In [42]:
UMLS_UTS_BASE = "https://uts-ws.nlm.nih.gov/rest"  # NLM UTS web services root URL for current UMLS

def _umls_api_get_semantic_types(cui: str, api_key: str) -> tuple[list[str], list[str]]:
    """Fetch semantic types for one CUI via UTS; returns parallel lists of TUI codes and type names."""

    def _extract(st_list):  # Inner helper to parse varying JSON shapes for semantic type arrays
        tuis, names = [], []  # Accumulators
        for st in st_list or []:  # st is one semantic type object from JSON
            uri = st.get("uri", "") or ""  # Often encodes .../TUI/###
            ui = st.get("ui", "") or ""  # Alternate field that may hold TUI
            nm = st.get("name", "") or ""  # Human-readable semantic type name
            code = None  # Extracted three-digit TUI string
            if "TUI/" in uri:
                code = uri.split("TUI/")[-1]  # Parse TUI from URI tail
            else:
                m = re.search(r"(T\d{3})", uri)  # Regex fallback on uri
                if m:
                    code = m.group(1)
                else:
                    m2 = re.search(r"(T\d{3})", ui)  # Regex on ui field
                    if m2:
                        code = m2.group(1)
            if code:
                tuis.append(code)
                names.append(nm)
        return tuis, names

    try:  # Primary endpoint: CUI content bundle often includes semanticTypes
        r = requests.get(f"{UMLS_UTS_BASE}/content/current/CUI/{cui}", params={"apiKey": api_key}, timeout=30)
        r.raise_for_status()  # Convert HTTP errors to exceptions
        tuis, names = _extract(r.json().get("result", {}).get("semanticTypes", []))
        if tuis:
            return tuis, names  # Success path
    except Exception:
        pass  # Fall through to secondary endpoint
    try:  # Secondary endpoint: dedicated semantictypes sub-resource
        r2 = requests.get(
            f"{UMLS_UTS_BASE}/content/current/CUI/{cui}/semantictypes",
            params={"apiKey": api_key},
            timeout=30,
        )
        r2.raise_for_status()
        j = r2.json()
        res = j.get("result")
        st_list = res.get("semanticTypes", []) if isinstance(res, dict) else res  # Normalize dict vs list shapes
        return _extract(st_list)
    except Exception:
        return [], []  # On total failure, empty lists imply unknown types


In [43]:
def _umls_api_resolve_disease(disease_norm: str, api_key: str, max_results: int = 25) -> str | None:
    """Query UTS search/current; keep disease-semantic-type hits; score like offline; return best CUI or None."""
    if requests is None:
        return None  # Cannot call API without requests installed
    url = f"{UMLS_UTS_BASE}/search/current"  # UTS search endpoint scoped to current UMLS release
    params = {"apiKey": api_key, "string": disease_norm, "pageSize": max_results, "searchType": "words"}  # words search: tokenized match
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    results = r.json().get("result", {}).get("results", [])  # List of hit dicts
    scored = []  # Collect (sort_key, cui) for ranking
    for x in results:  # Each hit is a candidate CUI
        ui = (x.get("ui") or "").strip()  # Candidate identifier string
        if not re.fullmatch(r"C\d+", ui):
            continue  # Keep only CUIs matching C + digits pattern
        tuis, _ = _umls_api_get_semantic_types(ui, api_key)  # Need TUIs to apply disease filter
        tset = set(tuis) if tuis else set()
        if not (tset & ALLOWED_DISEASE_STYS):
            continue  # Discard non-disease-like concepts
        hit_name = _norm_disease_label(x.get("name") or "")  # Normalize hit preferred string
        mth = _cui_mth_pn.get(ui)  # Local MTH PN if this CUI was in your extract
        mth_exact = bool(mth and mth == disease_norm)  # Strongest lexical signal
        name_exact = hit_name == disease_norm  # Exact match on returned name
        pref = disease_norm in _cui_pref_names_wide.get(ui, ())  # Preferred synonym match in local data
        sty_r = _sty_tiebreak_rank_for(tset)  # Lower is better; ties broken later by ui
        key = (-int(mth_exact), -int(name_exact), -int(pref), sty_r, ui)  # Tuple sort: True>False via negated ints
        scored.append((key, ui))
    if not scored:
        return None  # No acceptable disease hit
    scored.sort(key=lambda z: z[0])  # Ascending sort on key tuple
    return scored[0][1]  # Best CUI string


def resolve_umls_for_repurposed_disease_offline(disease_norm: str) -> str | None:
    """Resolve using only local _name_to_cuis_wide + MRSTY filters + scoring (no network)."""
    typed = [  # List comprehension: candidate CUIs that have at least one allowed disease TUI
        cui
        for cui in _name_to_cuis_wide.get(disease_norm, [])
        if _cui_tuis_full.get(cui, frozenset()) & ALLOWED_DISEASE_STYS
    ]
    if not typed:
        return None  # No disease-typed CUIs for this string
    best = None  # Winning CUI
    best_key = None  # Winning sort key tuple
    for cui in typed:  # Score each surviving CUI
        mth = _cui_mth_pn.get(cui)
        mth_exact = bool(mth and mth == disease_norm)
        pref = disease_norm in _cui_pref_names_wide.get(cui, ())
        key = (  # Offline tuple omits API name_exact; uses STY rank then stable CUI tie-break
            -int(mth_exact),
            -int(pref),
            _sty_tiebreak_rank_for(_cui_tuis_full.get(cui, frozenset())),
            cui,
        )
        if best_key is None or key < best_key:  # Lexicographic tuple comparison
            best_key = key
            best = cui
    return best  # Single best CUI or None


def resolve_umls_for_repurposed_disease(disease_norm: str) -> tuple[str | None, dict[str, object]]:
    """Try offline resolver; on failure optionally call UTS. Returns (cui, meta) where meta["used_api"] tracks source."""
    meta: dict[str, object] = {"used_api": False}  # Provenance flag for downstream QA logging
    u = resolve_umls_for_repurposed_disease_offline(disease_norm)
    if u is not None:
        return u, meta  # Offline hit
    _k = os.environ.get("UMLS_API_KEY", "").strip()  # UTS API key from environment
    if not _k:
        return None, meta  # No key → cannot call UTS
    try:
        u = _umls_api_resolve_disease(disease_norm, _k)
        if u is not None:
            meta["used_api"] = True  # Mark that network was used successfully
        return u, meta
    except Exception:
        return None, meta  # Swallow API errors into None result


In [44]:
# -----------------------------------------------------------------------------
# Build per-novel-pair rows: CAS from DrugBank (preferred) else DrugCentral `indication` CAS map.
# UMLS CUI from hybrid resolver; optional print tags for pluralization QA and API fallback visibility.
# -----------------------------------------------------------------------------
def split_pair(pair: str) -> tuple[str, str]:
    """Split the composite key on the first underscore only (drug names might contain underscores rarely)."""
    drug, disease = pair.split("_", 1)  # maxsplit=1 preserves disease text containing underscores
    return drug.strip(), disease.strip()  # Trim accidental spaces from lowercasing pipeline


def _suffix_qa_tag_for_disease_norm(dis_norm: str) -> str | None:
    """Return a short QA label for plural-looking endings, excluding Greek/Latin *is disease endings."""
    t = (dis_norm or "").strip()  # Defensive copy
    if len(t) < 2:
        return None  # Too short to mean anything for suffix heuristics
    if t.endswith("is"):
        return None  # Skip arthritis, fibrosis, etc.
    if t.endswith("es"):
        return "ENDS_ES"  # e.g. "diseases", "cancers" pattern flag
    if t.endswith("s") and not t.endswith("ss"):
        return "ENDS_S"  # e.g. "polyps", "lupus" (but not "class")
    return None  # No tag


_dc_cas_by_lower_name = (  # Series-derived dict: lowercased DrugCentral drug name -> cas_reg_no
    indication.dropna(subset=["name", "cas_reg_no"])  # Require both fields
    .assign(_nl=lambda x: x["name"].astype(str).str.strip().str.lower())  # Normalized lookup key column
    .drop_duplicates("_nl", keep="first")  # One CAS per normalized name (first row wins)
    .set_index("_nl")["cas_reg_no"]  # Series indexed by normalized name
    .to_dict()  # Plain dict for O(1) lookup in the loop
)

rows = []  # List of per-pair dict records accumulated for DataFrame construction
for pair in new_pairs:  # Only novel pairs need new identifier resolution
    drug_l, dis_l = split_pair(pair)  # Lowercased fragments from key
    db = drugbank_by_name.get(drug_l, {})  # DrugBank record or empty dict if unknown
    cas = db.get("cas_reg_no") or _dc_cas_by_lower_name.get(drug_l)  # Prefer XML CAS, else DrugCentral
    dbid = db.get("drugbank_id")  # Primary DrugBank id if parsed
    dis_norm = _norm_disease_label(dis_l)  # Align disease string with UMLS index keys
    umls, umeta = resolve_umls_for_repurposed_disease(dis_norm)  # (CUI or None, metadata)
    qa = _suffix_qa_tag_for_disease_norm(dis_norm)  # Optional QA tag
    if qa is not None:
        print("[{}]".format(qa), f"disease_norm={dis_norm!r}", f"cui={umls}", f"pair={pair!r}")  # Manual plural QA
    if umeta.get("used_api"):
        print("[UMLS_API]", f"disease_norm={dis_norm!r}", f"cui={umls}", f"pair={pair!r}")  # Network path marker
    rows.append(  # Accumulate one structured dict per pair
        {
            "pair": pair,
            "drug_name": drug_l,
            "disease_name": dis_l,
            "cas_reg_no": cas,
            "drugbank_id": dbid,
            "umls_cui": umls,
        }
    )


0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
1  -->  T191
2  -->  T048
3  -->  T019
4  -->  T020
5  -->  T046
0  -->  T047
1  -->  T191
2  -->  T048
3  -->  T019
4  -->  T020
5  -->  T046
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
1  -->  T191
2  -->  T048
3  -->  T019
4  -->  T020
5  -->  T046
0  -->  T047
0  -->  T047
0  -->  T047
1  -->  T191
2  -->  T048
0  -->  T047
1  -->  T191
2  -->  T048
0  -->  T047
1  -->  T191
2  -->  T048
0  -->  T047
1  -->  T191
2  -->  T048
0  -->  T047
1  -->  T191
2  -->  T048
0  -->  T047
1  -->  T191
2  -->  T048
0  -->  T047
1  -->  T191
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047
0  -->  T047

In [45]:
# Summarize how many rows received CAS and UMLS identifiers before any row-level filtering.
pair_ids = pd.DataFrame(rows)  # Materialize list of dicts as a DataFrame
print("CAS (DrugBank XML, name/synonym match):", pair_ids["cas_reg_no"].notna().sum(), "/", len(pair_ids))  # Non-null CAS count
print("UMLS (disease STY filter + tie-break):", pair_ids["umls_cui"].notna().sum(), "/", len(pair_ids))  # Non-null CUI count
pair_ids.head(10)  # Inspect first 10 mapped rows


CAS (DrugBank XML, name/synonym match): 463 / 531
UMLS (disease STY filter + tie-break): 522 / 531


,pair,drug_name,disease_name,cas_reg_no,drugbank_id,umls_cui
0,"acetylcysteine_kidney failure, acute",acetylcysteine,"kidney failure, acute",616-91-1,DB06151,C0022660
1,acitretin_psoriasis,acitretin,psoriasis,55079-83-9,DB00459,C0033860
2,aclidinium bromide_chronic obstructive airway ...,aclidinium bromide,chronic obstructive airway disease,320345-99-1,None,C0024117
3,adefovir dipivoxil_hepatitis b,adefovir dipivoxil,hepatitis b,142340-99-6,DB00718,C0019163
4,"adefovir dipivoxil_hepatitis b, chronic",adefovir dipivoxil,"hepatitis b, chronic",142340-99-6,DB00718,C0524909
5,adefovir_hepatitis b,adefovir,hepatitis b,106941-25-7,DB13868,C0019163
6,"adefovir_hepatitis b, chronic",adefovir,"hepatitis b, chronic",106941-25-7,DB13868,C0524909
7,adenosine_atherosclerosis,adenosine,atherosclerosis,58-61-7,DB00640,C0004153
8,adenosine_coronary artery disease,adenosine,coronary artery disease,58-61-7,DB00640,C1956346
9,albuterol sulfate_asthma,albuterol sulfate,asthma,None,None,C0004096


In [46]:
# pandas summary: dtypes, non-null counts, memory usage — catches object columns that should be numeric.
pair_ids.info()  # Side effect: prints to stdout; returns None


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 531 entries, 0 to 530
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   pair          531 non-null    object
 1   drug_name     531 non-null    object
 2   disease_name  531 non-null    object
 3   cas_reg_no    463 non-null    object
 4   drugbank_id   456 non-null    object
 5   umls_cui      522 non-null    object
dtypes: object(6)
memory usage: 25.0+ KB


In [47]:
# Full table display in Jupyter (can be large; optional validation cell).
pair_ids  # Renders HTML table in notebook frontends


,pair,drug_name,disease_name,cas_reg_no,drugbank_id,umls_cui
0,"acetylcysteine_kidney failure, acute",acetylcysteine,"kidney failure, acute",616-91-1,DB06151,C0022660
1,acitretin_psoriasis,acitretin,psoriasis,55079-83-9,DB00459,C0033860
2,aclidinium bromide_chronic obstructive airway ...,aclidinium bromide,chronic obstructive airway disease,320345-99-1,None,C0024117
3,adefovir dipivoxil_hepatitis b,adefovir dipivoxil,hepatitis b,142340-99-6,DB00718,C0019163
4,"adefovir dipivoxil_hepatitis b, chronic",adefovir dipivoxil,"hepatitis b, chronic",142340-99-6,DB00718,C0524909
...,...,...,...,...,...,...
526,"zoledronic acid_osteoporosis, postmenopausal",zoledronic acid,"osteoporosis, postmenopausal",118072-93-8,DB00399,C0029458
527,zolmitriptan_common migraine,zolmitriptan,common migraine,139264-17-8,DB00315,C0338480
528,zolpidem_sleep initiation and maintenance diso...,zolpidem,sleep initiation and maintenance disorders,82626-48-0,DB00425,C0021603
529,zonisamide_epilepsy,zonisamide,epilepsy,68291-97-4,DB00909,C0014544


In [48]:
# Manual override example: force a specific CUI for one disease label after visual inspection.
pair_ids.loc[pair_ids["disease_name"] == "nasal polyps", "umls_cui"] = "C0027430"  # .loc avoids chained assignment


In [49]:
pair_ids[pair_ids.disease_name.str.lower().str.contains("nasal", case = False)]  # Filter rows for substring "nasal" case-insensitive


,pair,drug_name,disease_name,cas_reg_no,drugbank_id,umls_cui
74,budesonide_nasal polyps,budesonide,nasal polyps,51333-22-3,DB01222,C0027430


In [50]:
# Export-quality filter: require all three identifiers and exclude literal string "None" artifacts.
pair_ids = pair_ids[  # Boolean mask combined with & (parentheses required for operator precedence)
    (pair_ids["cas_reg_no"].notna()) &
    (pair_ids["umls_cui"].notna()) &
    (pair_ids["drugbank_id"].notna()) &
    (pair_ids["cas_reg_no"] != "None") &
    (pair_ids["umls_cui"] != "None")
]


In [51]:
# Ensure uniqueness at the drug–disease label grain after mapping (duplicate keys can appear upstream).
pair_ids = pair_ids.drop_duplicates(subset=["drug_name", "disease_name"])  # Keeps first duplicate row
len(pair_ids)  # Row count after deduplication


448

In [52]:
# Distinct (disease_name, umls_cui) pairs for spot-checking mapping consistency across drugs.
temp = pair_ids[["disease_name", "umls_cui"]]  # Project two columns
temp = temp.drop_duplicates()  # Unique combinations for QA sheet


In [53]:
# Filter the QA table to one disease string of interest (edit the literal to match your review list).
temp[temp.disease_name == "ureteral calculi"]  # Expect one row if label exists after dedupe


,disease_name,umls_cui
466,ureteral calculi,C1456865


In [54]:
# Expected CUI for "ureteral calculi" in this notebook’s manual check (verify in UMLS browser / MTH).
# ureteral calculi --> C1456865


In [55]:
# Reduce to DrugCentral-like edge columns: chemical CAS, disease UMLS CUI, and fixed relationship label.
pair_ids = pair_ids[["cas_reg_no", "umls_cui"]]  # Drop drug/disease text columns for merge-ready file
pair_ids = pair_ids.query("not @pair_ids.cas_reg_no.isna()")  # pandas query: drop rows with missing CAS
pair_ids = pair_ids.query("not @pair_ids.umls_cui.isna()")  # Drop rows with missing CUI
pair_ids["relationship_name"] = "indication"  # Constant column to align with DrugCentral relationship_name
pair_ids  # Preview final edge list


,cas_reg_no,umls_cui,relationship_name
0,616-91-1,C0022660,indication
1,55079-83-9,C0033860,indication
3,142340-99-6,C0019163,indication
4,142340-99-6,C0524909,indication
5,106941-25-7,C0019163,indication
...,...,...,...
526,118072-93-8,C0029458,indication
527,139264-17-8,C0338480,indication
528,82626-48-0,C0021603,indication
529,68291-97-4,C0014544,indication


In [56]:
# Some drugs/diseases may still collapse to identical CAS/CUI pairs — dedupe on those identifiers.
pair_ids = pair_ids.drop_duplicates()  # Default: all columns must match
len(pair_ids)  # Final edge count


448

In [57]:
# Persist the edge list next to this notebook; change path if you need a dated archive location.
pair_ids.to_csv(PREP_DIR / "outputs/20260405-RepurposedDrug_Indication.csv", index=False)  # index=False omits the DataFrame index column
